# Werkcollege-opdrachten Week 1.3

## Dependencies importeren

Kopieer in het codeblok hieronder van het vorige werkcollege de import-code voor de dependencies die het vaakst worden gebruikt om data in te lezen. Geef er ook de gebruikelijke aliassen aan.<br>
Zet eventuele warnings uit.

In [ ]:
import pandas as pd
import sqlite3
import warnings
warnings.simplefilter('ignore')

Zet de volgende bestanden in een makkelijk te vinden map, als je dat nog niet gedaan hebt:
- go_sales.sqlite
- go_crm.sqlite
- go_staff.sqlite

## Databasetabellen inlezen

Kopieer in het codeblok hieronder van het vorige werkcollege de code om een connectie met het bestand go_sales.sqlite te maken.

In [57]:
conn = sqlite3.connect('Data/go_sales.sqlite')

Lees van de ingelezen go_sales-database te volgende tabellen in met behulp van "SELECT * FROM *tabel*".
- product
- product_type
- product_line
- sales_staff
- sales_branch
- retailer_site
- country
- order_header
- order_details
- returned_item
- return_reason

In [6]:
product = pd.read_sql_query("SELECT * FROM product", conn)
product_type = pd.read_sql_query("SELECT * FROM product_type", conn)
product_line = pd.read_sql_query("SELECT * FROM product_line", conn)
sales_staff = pd.read_sql_query("SELECT * FROM sales_staff", conn)
sales_branch = pd.read_sql_query("SELECT * FROM sales_branch", conn)
retailer_site = pd.read_sql_query("SELECT * FROM retailer_site", conn)
country = pd.read_sql_query("SELECT * FROM country", conn)
order_header = pd.read_sql_query("SELECT * FROM order_header", conn)
order_details = pd.read_sql_query("SELECT * FROM order_details", conn)
returned_item = pd.read_sql_query("SELECT * FROM returned_item", conn)
return_reason = pd.read_sql_query("SELECT * FROM return_reason", conn)



Krijg je een "no such table" error? Dan heb je misschien met .connect() per ongeluk een leeg  databasebestand (.sqlite) aangemaakt.<br> <u>Let op:</u> lees eventueel de informatie uit het vorige werkcollege nog eens goed door.

Als je tijdens onderstaande opdrachten uit het oog verliest welke tabellen er allemaal zijn, kan je deze Pythoncode uitvoeren:

In [ ]:
sql_query = "SELECT * FROM sqlite_master WHERE type='table';"
#Vul dit codeblok verder in
pd.read_sql(sql_query, ...)
#Op de puntjes hoort de connectie naar go_sales óf go_staff óf go_crm te staan.

Let op! Voor alle onderstaande opdrachten mag je <u>alleen Python</u> gebruiken, <u>geen SQL!</u>

## Selecties op één tabel zonder functies

Geef een overzicht met daarin de producten en hun productiekosten waarvan de productiekosten lager dan 100 dollar en hoger dan 50 dollar ligt. (2 kolommen, 23 rijen)

In [ ]:
product_overzicht = product.loc[
    (product['PRODUCTION_COST'] > 50) & (product['PRODUCTION_COST'] < 100), 
    ['PRODUCT_NAME', 'PRODUCTION_COST']
]

product_overzicht

float64


Geef een overzicht met daarin de producten en hun marge waarvan de marge lager dan 20 % of hoger dan 60 % ligt. (2 kolommen, 7 rijen) 

In [ ]:
marge_overzicht = product.loc[
    (product['MARGIN'] < 0.20) | (product['MARGIN'] > 0.60), 
    ['PRODUCT_NAME', 'MARGIN']
]

marge_overzicht

Geef een overzicht met daarin de landen waar met francs wordt betaald. Sorteer de uitkomst op land.  (1 kolom, 3 rijen)

In [10]:
land_francs = country.loc[country['CURRENCY_NAME'] == 'francs', ['COUNTRY']].sort_values(by='COUNTRY')

land_francs

,COUNTRY
15,Belgium
0,France
7,Switzerland


Geef een overzicht met daarin de verschillende introductiedatums waarop producten met meer dan 50% marge worden geïntroduceerd (1 kolom, 7 rijen) 

In [ ]:
introductiedatums_overzicht = product.loc[product['MARGIN'] > 0.50,['INTRODUCTION_DATE']].drop_duplicates()
introductiedatums_overzicht

,INTRODUCTION_DATE
66,2000-10-26
85,1995-02-15
101,2003-12-10
103,2003-12-18
104,2003-12-27
106,2004-01-13
110,2003-12-15


Geef een overzicht met daarin het eerste adres en de stad van verkoopafdelingen waarvan zowel het tweede adres als de regio bekend is (2 kolommen, 7 rijen)

In [ ]:
verkoopafdelingen = sales_branch.loc[
    (sales_branch['ADDRESS2'].notna()) & (sales_branch['REGION'].notna()), 
    ['ADDRESS1', 'CITY']
]

verkoopafdelingen

Geef een overzicht met daarin de landen waar met dollars (dollars of new dollar) wordt betaald. Sorteer de uitkomst op land. (1 kolom, 4 rijen) 

In [ ]:
landen_dollars = country.loc[
    (country['CURRENCY_NAME'] == 'dollars') | (country['CURRENCY_NAME'] == 'new dollar'), 
    ['COUNTRY']
].sort_values(by='COUNTRY')

landen_dollars

,COUNTRY
14,Australia
3,Canada
11,Taiwan
2,United States


Geef een overzicht met daarin beide adressen en de stad van vestigingen van klanten waarvan de postcode begint met een ‘D’ (van duitsland). Filter op vestigingen die een tweede adres hebben. (3 kolommen, 2 rijen) 

In [15]:
duitse_vestigingen = retailer_site.loc[
    (retailer_site['POSTAL_ZONE'].str[0] == 'D') & (retailer_site['ADDRESS2'].notna()),
    ['ADDRESS1', 'ADDRESS2', 'CITY']
]

duitse_vestigingen

,ADDRESS1,ADDRESS2,CITY
31,Röntgenstraße 90,3. Tür rechts,Frankfurt
59,Grubesallee 141,4. Stock,Hamburg


## Selecties op één tabel met functies

Geef het totaal aantal producten dat is teruggebracht (1 waarde) 

In [16]:
totaal = returned_item['RETURN_QUANTITY'].sum()

totaal

np.int64(11908)

Geef het aantal regio’s waarin verkoopafdelingen gevestigd zijn. (1 waarde)

In [18]:
aantal_regios = sales_branch['REGION'].drop_duplicates().count()

aantal_regios

np.int64(15)

Maak 3 variabelen:
- Een met de laagste
- Een met de hoogste
- Een met de gemiddelde (afgerond op 2 decimalen)

marge van producten (3 kolommen, 1 rij) 

In [20]:
laagste_marge = product['MARGIN'].min()
hoogste_marge = product['MARGIN'].max()

gemiddelde_marge = round(product['MARGIN'].mean(), 2)

marge_overzicht = pd.DataFrame({
    'LAAGSTE_MARGE': [laagste_marge],
    'HOOGSTE_MARGE': [hoogste_marge],
    'GEMIDDELDE_MARGE': [gemiddelde_marge]
})

marge_overzicht

,LAAGSTE_MARGE,HOOGSTE_MARGE,GEMIDDELDE_MARGE
0,0.17,0.7,0.4


Geef het aantal vestigingen van klanten waarvan het 2e adres niet bekend is (1 waarde)

In [21]:
aantal_onbekend_adres2 = retailer_site.loc[retailer_site['ADDRESS2'].isna(), 'ADDRESS1'].count()

aantal_onbekend_adres2

np.int64(286)

Geef de gemiddelde kostprijs van de verkochte producten waarop korting (unit_sale_price < unit_price) is verleend (1 waarde) 

In [22]:
korting_selectie = order_details.loc[order_details['UNIT_SALE_PRICE'] < order_details['UNIT_PRICE']]

gemiddelde_kostprijs = korting_selectie['UNIT_COST'].mean()

gemiddelde_kostprijs

np.float64(127.34087927598803)

Geef een overzicht met daarin het aantal medewerkers per medewerkersfunctie (2 kolommen, 7 rijen) 

In [23]:
medewerkers_per_functie = sales_staff.groupby('POSITION_EN')[['SALES_STAFF_CODE']].count()

medewerkers_per_functie = medewerkers_per_functie.rename(columns={'SALES_STAFF_CODE': 'AANTAL_MEDEWERKERS'})

medewerkers_per_functie

,AANTAL_MEDEWERKERS
POSITION_EN,
Branch Manager,19
District Manager,4
General Manager,1
Level 1 Sales Representative,13
Level 2 Sales Representative,39
Level 3 Sales Representative,24
Regional Manager,2


Geef een overzicht met daarin per telefoonnummer het aantal medewerkers dat op dat telefoonnummer bereikbaar is. Toon alleen de telefoonnummer waarop meer dan 4 medewerkers bereikbaar zijn. (2 kolommen, 10 rijen) 

In [24]:
tel_overzicht = sales_staff.groupby('WORK_PHONE')[['SALES_STAFF_CODE']].count()

tel_overzicht = tel_overzicht.rename(columns={'SALES_STAFF_CODE': 'AANTAL_MEDEWERKERS'})

resultaat = tel_overzicht.loc[tel_overzicht['AANTAL_MEDEWERKERS'] > 4]

resultaat

,AANTAL_MEDEWERKERS
WORK_PHONE,
+(44) 121 3505267,5
+(61) 03 2982 4242,5
+31 (0)20 692 93 94,5
+33 1 68 94 52 20,5
1 (206) 292-0012,5
1 (305) 557-4810,6
1 (310) 281-5722,5
1 (403) 232-5986,5
1 (416) 493-5595,5


## Selecties op meerdere tabellen zonder functies

Geef een overzicht met daarin het eerste adres en de stad van vestigingen van klanten uit ‘Netherlands’ (2 kolommen, 20 rijen) 

In [25]:
vestigingen_landen = pd.merge(retailer_site, country, on='COUNTRY_CODE')

nederlandse_vestigingen = vestigingen_landen.loc[
    vestigingen_landen['COUNTRY'] == 'Netherlands', 
    ['ADDRESS1', 'CITY']
]

nederlandse_vestigingen

,ADDRESS1,CITY
240,Beets 36,Beets
241,Perestraat 1,Deventer
242,Laan van Meerdervoort 966,Den Haag
243,Kalfjeslaan 47,Amstelveen
244,Startbaan 11,Amstelveen
245,Noorderspoorsingel 28,Groningen
246,Hogehilweg 13,Amsterdam
247,Spoorstraat 43,Varsseveld
248,Westhavenweg 8,Amsterdam
249,Westzeedijk 4,Muiden


Geef een overzicht met daarin de productnamen die tot het producttype ‘Eyewear’ behoren. (1 kolom, 5 rijen) 

In [26]:
product_info = pd.merge(product, product_type, on='PRODUCT_TYPE_CODE')

eyewear_producten = product_info.loc[
    product_info['PRODUCT_TYPE_EN'] == 'Eyewear', 
    ['PRODUCT_NAME']
]

eyewear_producten

,PRODUCT_NAME
67,Polar Sun
68,Polar Ice
69,Polar Sports
70,Polar Wave
71,Polar Extreme


Geef een overzicht met daarin alle unieke eerste adressen van klantvestigingen en de voornaam en achternaam van de verkopers die ‘Branch Manager’ zijn en aan deze vestigingen hebben verkocht (3 kolommen, 1 rij) 

In [32]:
staff_orders = pd.merge(sales_staff, order_header, on='SALES_STAFF_CODE')

alles_samen = pd.merge(staff_orders, retailer_site, on='RETAILER_SITE_CODE')

resultaat = alles_samen.loc[
    alles_samen['POSITION_EN'] == 'Branch Manager', 
    ['ADDRESS1', 'FIRST_NAME', 'LAST_NAME']
].drop_duplicates().head(1)

resultaat

,ADDRESS1,FIRST_NAME,LAST_NAME
3727,"Alameda Santos, 9876",Bayard,Lopes


Geef een overzicht met daarin van de verkopers hun functie en indien zij iets hebben verkocht de datum waarop de verkoop heeft plaatsgevonden. Laat alleen de verschillende namen van de posities zien van de verkopers die het woord ‘Manager’ in hun positienaam hebben staan. (2 kolommen, 7 rijen) 

In [33]:
verkoop_overzicht = pd.merge(sales_staff, order_header, on='SALES_STAFF_CODE', how='left')

manager_selectie = verkoop_overzicht.loc[
    verkoop_overzicht['POSITION_EN'].str.contains('Manager', na=False)
]

resultaat = manager_selectie[['POSITION_EN', 'ORDER_DATE']].drop_duplicates().head(7)

resultaat

,POSITION_EN,ORDER_DATE
0,Branch Manager,NaN
320,Regional Manager,NaN
648,District Manager,NaN
3748,Branch Manager,2024-09-09
3749,Branch Manager,2024-03-13
3750,Branch Manager,2024-01-09
4186,General Manager,NaN


Geef een overzicht met daarin de verschillende namen van producten en bijbehorende namen van producttypen van de producten waarvoor ooit meer dan 750 stuks tegelijk verkocht zijn. (2 kolommen, 9 rijen) 

In [34]:
stap1 = pd.merge(order_details, product, on='PRODUCT_NUMBER')

volledige_tabel = pd.merge(stap1, product_type, on='PRODUCT_TYPE_CODE')

grote_verkopen = volledige_tabel.loc[volledige_tabel['QUANTITY'] > 750]

resultaat = grote_verkopen[['PRODUCT_NAME', 'PRODUCT_TYPE_EN']].drop_duplicates()

resultaat

,PRODUCT_NAME,PRODUCT_TYPE_EN
539,BugShield Extreme,Insect Repellents
1759,Star Peg,Tents
20121,BugShield Lotion Lite,Insect Repellents
20147,Sun Shelter 15,Sunscreen
20152,Sun Shelter 30,Sunscreen
31236,BugShield Spray,Insect Repellents
37272,Firefly 2,Lanterns
37348,BugShield Lotion,Insect Repellents
37363,Sun Shield,Sunscreen


Geef een overzicht met daarin de productnamen waarvan ooit meer dan 40% korting is verleend. De formule voor korting is: (unit_price - unit_sale_price) / unit_price (1 kolom, 8 rijen) 

In [35]:
product_korting = pd.merge(product, order_details, on='PRODUCT_NUMBER')

resultaat = product_korting.loc[
    ((product_korting['UNIT_PRICE'] - product_korting['UNIT_SALE_PRICE']) / product_korting['UNIT_PRICE']) > 0.40,
    ['PRODUCT_NAME']
]

eindresultaat = resultaat.drop_duplicates()

eindresultaat

,PRODUCT_NAME
27830,BugShield Natural
28283,BugShield Spray
28694,BugShield Lotion Lite
29127,BugShield Lotion
29464,BugShield Extreme
30562,Sun Shelter 15
30929,Sun Shelter 30
34441,Hailstorm Titanium Woods Set


Geef een overzicht met daarin de retourreden van producten waarvan ooit meer dan 90% van de aangeschafte hoeveelheid is teruggebracht (return_quantity/quantity). (1 kolom, 3 rijen) 

In [39]:
return_item = pd.read_sql_query("SELECT * FROM returned_item", conn) 
order_details = pd.read_sql_query("SELECT * FROM order_details", conn)
return_reason = pd.read_sql_query("SELECT * FROM return_reason", conn)

retouren_met_details = pd.merge(return_item, order_details, on='ORDER_DETAIL_CODE')
retouren_volledig = pd.merge(retouren_met_details, return_reason, on='RETURN_REASON_CODE')

selectie_redenen = retouren_volledig.loc[
    (retouren_volledig['RETURN_QUANTITY'] / retouren_volledig['QUANTITY']) > 0.90,
    ['RETURN_DESCRIPTION_EN']
]

unieke_redenen = selectie_redenen.drop_duplicates()
unieke_redenen

,RETURN_DESCRIPTION_EN
0,Unsatisfactory product
2,Wrong product shipped
3,Wrong product ordered


## Selecties op meerdere tabellen met functies

Geef een overzicht met daarin per producttype het aantal producten die tot dat producttype behoren. (2 kolommen, 21 rijen) 

In [41]:
product_met_type = pd.merge(product, product_type, on='PRODUCT_TYPE_CODE')

overzicht = product_met_type.groupby('PRODUCT_TYPE_EN')[['PRODUCT_NUMBER']].count()

overzicht = overzicht.rename(columns={'PRODUCT_NUMBER': 'AANTAL_PRODUCTEN'})

overzicht

,AANTAL_PRODUCTEN
PRODUCT_TYPE_EN,
Binoculars,4
Climbing Accessories,7
Cooking Gear,10
Eyewear,5
First Aid,5
Golf Accessories,4
Insect Repellents,5
Irons,4
Knives,5


Geef een overzicht met daarin per land het aantal vestigingen van klanten die zich in dat land bevinden. (2 kolommen, 21 rijen) 

In [43]:

vestigingen_landen = pd.merge(retailer_site, country, on='COUNTRY_CODE')

overzicht_landen = vestigingen_landen.groupby('COUNTRY')[['RETAILER_SITE_CODE']].count()

overzicht_landen = overzicht_landen.rename(columns={'RETAILER_SITE_CODE': 'AANTAL_VESTIGINGEN'})

overzicht_landen

,AANTAL_VESTIGINGEN
COUNTRY,
Australia,10
Austria,10
Belgium,10
Brazil,1
Canada,31
China,10
Denmark,6
Finland,5
France,29


Geef een overzicht met daarin van de producten behorend tot het producttype ‘Cooking Gear’ per productnaam de totaal verkochte hoeveelheid en de gemiddelde verkoopprijs. Sorteer de uitkomst op totaal verkochte hoeveelheid. (6 kolommen, 10 rijen)

In [44]:
product_met_type = pd.merge(product, product_type, on='PRODUCT_TYPE_CODE')
volledige_data = pd.merge(product_met_type, order_details, on='PRODUCT_NUMBER')

cooking_gear_data = volledige_data.loc[volledige_data['PRODUCT_TYPE_EN'] == 'Cooking Gear']

overzicht = cooking_gear_data.groupby('PRODUCT_NAME').agg({
    'QUANTITY': 'sum',
    'UNIT_SALE_PRICE': 'mean',
    'UNIT_PRICE': 'first',
    'UNIT_COST': 'first',
    'MARGIN': 'first'
})

resultaat = overzicht.reset_index().sort_values(by='QUANTITY', ascending=False)

eindresultaat = resultaat.head(10)

eindresultaat

,PRODUCT_NAME,QUANTITY,UNIT_SALE_PRICE,UNIT_PRICE,UNIT_COST,MARGIN
9,TrailChef Water Bag,36738,6.032506,6.59,4.38,0.33
2,TrailChef Cup,27418,6.976490,7.32,5.23,0.28
0,TrailChef Canteen,26544,11.979418,12.53,9.64,0.23
6,TrailChef Kitchen Kit,22206,22.345975,23.80,17.25,0.28
8,TrailChef Utensils,13776,17.294151,19.29,11.35,0.40
4,TrailChef Double Flame,12476,132.672882,151.77,88.23,0.41
1,TrailChef Cook Set,10878,51.687926,54.93,38.40,0.30
5,TrailChef Kettle,8826,12.201235,13.22,9.93,0.25
7,TrailChef Single Flame,8504,63.827622,66.77,48.38,0.28
3,TrailChef Deluxe Cook Set,3110,121.217638,129.72,92.00,0.28


Geef een overzicht met daarin per land de naam van het land, de naam van de stad waar de verkoopafdeling is gevestigd (noem de kolomnaam in het overzicht ‘verkoper’) en het aantal steden waar zich klanten bevinden in dat land (noem de kolomnaam in het overzicht ‘klanten’) (3 kolommen, 29 rijen) 

In [45]:
klanten_per_land = pd.merge(retailer_site, country, on='COUNTRY_CODE')
klanten_overzicht = klanten_per_land.groupby('COUNTRY')[['CITY']].nunique().reset_index()
klanten_overzicht = klanten_overzicht.rename(columns={'CITY': 'klanten'})

verkopers_per_land = pd.merge(sales_branch, country, on='COUNTRY_CODE')
verkopers_overzicht = verkopers_per_land[['COUNTRY', 'CITY']].rename(columns={'CITY': 'verkoper'})

resultaat = pd.merge(verkopers_overzicht, klanten_overzicht, on='COUNTRY')

eindresultaat = resultaat[['COUNTRY', 'verkoper', 'klanten']].head(29)

eindresultaat

,COUNTRY,verkoper,klanten
0,France,Paris,14
1,Italy,Milano,9
2,Netherlands,Amsterdam,15
3,Germany,Hamburg,18
4,Germany,München,18
5,Sweden,Kista,10
6,Canada,Calgary,16
7,Canada,Toronto,16
8,United States,Boston,62
9,United States,Seattle,62


## Pythonvertalingen van SUBSELECT en UNION met o.a. for-loops

Geef een overzicht met daarin de voornaam en de achternaam van de medewerkers die nog nooit wat hebben verkocht (2 kolommen, 25 rijen) 

In [47]:
staff_orders = pd.merge(sales_staff, order_header, on='SALES_STAFF_CODE', how='left')

niet_verkocht = staff_orders.loc[staff_orders['ORDER_NUMBER'].isna()]

resultaat = niet_verkocht[['FIRST_NAME', 'LAST_NAME']]

eindresultaat = resultaat.head(25)

eindresultaat

,FIRST_NAME,LAST_NAME
0,Denis,Pagé
320,Else,Mörike
321,Frank,Fuchs
647,Fritz,Hirsch
648,Jörg,Kunze
728,Maria,Iacobucci
946,Kick,Kalkman
1309,Karin,Bergström
1395,Sally,White
1421,Frank,Bretton


Geef een overzicht met daarin het aantal producten waarvan de marge lager is dan de gemiddelde marge van alle producten samen. Geef in het overzicht tevens aan wat de gemiddelde marge is van dit aantal producten waarvan de marge lager dan de gemiddelde marge van alle producten samen is. (1 kolom, 2 rijen) 

In [48]:
gemiddelde_totaal = product['MARGIN'].mean()

selectie_laag = product.loc[product['MARGIN'] < gemiddelde_totaal]

aantal = selectie_laag['PRODUCT_NUMBER'].count()
gemiddelde_selectie = selectie_laag['MARGIN'].mean()

overzicht = pd.DataFrame([aantal, gemiddelde_selectie], columns=['Marge_Statistieken'])

overzicht

,Marge_Statistieken
0,59.000000
1,0.292203


Geef een overzicht met daarin de namen van de producten die voor meer dan 500 (verkoopprijs) zijn verkocht maar nooit zijn teruggebracht. (1 kolom, 13 rijen) 

In [51]:
product_sales = pd.merge(product, order_details, on='PRODUCT_NUMBER')

dure_producten = product_sales.loc[product_sales['UNIT_SALE_PRICE'] > 500]

check_retouren = pd.merge(dure_producten, return_item, on='ORDER_DETAIL_CODE', how='left')

nooit_retour = check_retouren.loc[check_retouren['RETURN_REASON_CODE'].isna()]

resultaat = nooit_retour[['PRODUCT_NAME']].drop_duplicates().head(13)

resultaat

,PRODUCT_NAME
0,Star Dome
397,Star Gazer 2
575,Star Gazer 3
961,Star Gazer 6
1284,Husky Rope 200
1450,Hailstorm Steel Irons
1761,Hailstorm Titanium Irons
2125,Lady Hailstorm Steel Irons
2286,Lady Hailstorm Titanium Irons
2549,Hailstorm Titanium Woods Set


Geef een overzicht met daarin per (achternaam van) medewerker of hij/zij manager is of niet, door deze informatie toe te voegen als extra 'Ja/Nee'-kolom.<br>
Hint: gebruik een for-loop waarin je o.a. bepaalt of het woord 'Manager' in de functie (position_en) staat. (2 kolommen, 102 rijen).

In [52]:
manager_status = []

for index, rij in sales_staff.iterrows():
  
    if 'Manager' in str(rij['POSITION_EN']):
        manager_status.append('Ja')
    else:
        manager_status.append('Nee')

sales_staff['Manager_Status'] = manager_status

overzicht_managers = sales_staff[['LAST_NAME', 'Manager_Status']]

overzicht_managers

,LAST_NAME,Manager_Status
0,Pagé,Ja
1,Michel,Nee
2,Clermont,Nee
3,Jauvin,Nee
4,Wiesinger,Nee
...,...,...
97,Laermans,Nee
98,De Crée,Nee
99,Lattrez,Nee
100,Seefelder,Nee


Met de onderstaande code laat je Python het huidige jaar uitrekenen.

In [2]:
from datetime import date
date.today().year

2026

Met de onderstaande code selecteer je op een bepaald jaartal uit een datum.

In [3]:
from datetime import datetime

date_str = '16-8-2013'
date_format = '%d-%m-%Y'
date_obj = datetime.strptime(date_str, date_format)

date_obj.year

2013

Geef met behulp van bovenstaande hulpcode een overzicht met daarin op basis van het aantal jaar dat iemand in dienst is of een medewerker ‘kort in dienst’ (minder dan 30 jaar in dienst) of een ‘lang in dienst’ (groter dan gelijk aan 30 jaar in dienst) is. Geef daarbij per medewerker in een aparte kolom zowel ‘kort in dienst’ als ‘lang in dienst’ aan. Gebruik (wederom) een for-loop.<br>
(2 kolommen, 102 rijen) 

In [55]:
import datetime

huidig_jaar = 2026

diensttijd_labels = []

for index, rij in sales_staff.iterrows():

    datum_waarde = rij.get('hire_date') or rij.get('HIRE_DATE')
    
    jaar_indienst = pd.to_datetime(datum_waarde).year
    jaren_in_dienst = huidig_jaar - jaar_indienst
    
    if jaren_in_dienst < 30:
        diensttijd_labels.append('kort in dienst')
    else:
        diensttijd_labels.append('lang in dienst')

sales_staff['Diensttijd_Status'] = diensttijd_labels
overzicht = sales_staff[['LAST_NAME', 'Diensttijd_Status']]

overzicht.head(10)

,LAST_NAME,Diensttijd_Status
0,Pagé,lang in dienst
1,Michel,lang in dienst
2,Clermont,lang in dienst
3,Jauvin,lang in dienst
4,Wiesinger,lang in dienst
5,Mörike,lang in dienst
6,Fuchs,lang in dienst
7,Erler,lang in dienst
8,Winkler,lang in dienst
9,Hirsch,lang in dienst
